# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('east')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[1 0]'; Monster is at '[2 1]'
.....
H....
.M...
.....
.....
Hunter chose action 'north'
Monster chose action 'stay'


Step 2: Hunter is at '[0 0]'; Monster is at '[2 1]'
H....
.....
.M...
.....
.....
Hunter chose action 'west'
Monster chose action 'stay'


Step 3: Hunter is at '[0 0]'; Monster is at '[2 1]'
H....
.....
.M...
.....
.....
Hunter chose action 'east'
Monster chose action 'north'


Step 4: Hunter is at '[0 1]'; Monster is at '[1 1]'
.H...
.M...
.....
.....
.....
Hunter chose action 'north'
Monster chose action 'stay'


Step 5: Hunter is at '[0 1]'; Monster is at '[1 1]'
.H...
.M...
.....
.....
.....
Hunter chose action 'listen'
Monster chose action 'north'


Step 6: Hunter is at '[0 1]'; Monster is at '[0 1]'
.*...
.....
.....
.....
.....
Hunter caught the monster in 5 steps!


5

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [7]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[95, nan, 80, 36, nan, nan, nan, 99, 54, nan, 57, 87, nan, 37, nan, nan, nan, nan, 61, nan, nan, 58, nan, 16, nan, 36, 25, 52, 14, nan, nan, nan, 35, 90, 45, 44, 2, nan, nan, 26, 68, 64, nan, 5, 25, nan, nan, 4, 9, 15, 90, nan, 91, 95, 4, 0, nan, nan, nan, nan, nan, 56, nan, nan, nan, nan, nan, nan, nan, nan, 12, nan, 20, nan, nan, nan, 25, 65, nan, 47, 2, 61, 75, 30, 4, 49, 76, nan, 86, 44, 42, 53, 26, 9, nan, nan, nan, 29, nan, 4]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [8]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 46 times.
Average steps to catch the monster over 54 successful runs: 43.22


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [9]:
# here goes your agent implementation
def model_based_hunter_agent_function(hunter_location, monster_location):
    #initial state cache for first run
    if not hasattr(model_based_hunter_agent_function,"state"):
        model_based_hunter_agent_function.state = {
            "last_monster_location":None 
        }
    state = model_based_hunter_agent_function.state
    #Check if monster location is avaiable or not
    if monster_location is not None:
        state['last_monster_location'] = monster_location.copy()
    #If there are no monster position. Hunter have to listen to get location
    if (state['last_monster_location'] is None) or np.array_equal(hunter_location,state['last_monster_location']):
        return 'listen'

    #Calculate the distance
    target = state['last_monster_location']
    x = target[1] - hunter_location[1]
    y = target[0] - hunter_location[0]

    #quick have larger value then priority it to go first
    if abs(x) >= abs(y):
        if x < 0:
            return 'west'
        elif x > 0:
            return 'east'
    else:
        if y < 0:
            return 'north'
        elif y > 0:
            return 'south'

    return 'listen'

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [27]:
# Your experimentation code goes here
grid_size = 30
max_simulation_steps = 500
runs = 100
steps_taken = []
for _ in range(runs):
    #xóa state trước đó.
    if hasattr(model_based_hunter_agent_function,"state"): delattr(model_based_hunter_agent_function,"state")
    steps = arena_environment(model_based_hunter_agent_function,monster_agent_function_simple,n=grid_size,max_steps=max_simulation_steps,visualize=False,animation=False) #Tắt Visualize và animation để ra thẳng kết quả.
    steps_taken.append(steps)
    print(f"Result {_ + 1}: took {steps} steps to catch monster!")
print(f"Hunter failed {np.sum(np.isnan(steps_taken))} times.")
print(f"Average steps to catch the monster over {np.sum(~np.isnan(steps_taken))} successful runs: {round(np.nanmean(steps_taken), 2)} steps!")

Result 1: took 17 steps to catch monster!
Result 2: took 20 steps to catch monster!
Result 3: took 39 steps to catch monster!
Result 4: took 30 steps to catch monster!
Result 5: took 17 steps to catch monster!
Result 6: took 29 steps to catch monster!
Result 7: took 9 steps to catch monster!
Result 8: took 15 steps to catch monster!
Result 9: took 10 steps to catch monster!
Result 10: took 27 steps to catch monster!
Result 11: took 15 steps to catch monster!
Result 12: took 36 steps to catch monster!
Result 13: took 6 steps to catch monster!
Result 14: took 45 steps to catch monster!
Result 15: took 20 steps to catch monster!
Result 16: took 23 steps to catch monster!
Result 17: took 4 steps to catch monster!
Result 18: took 17 steps to catch monster!
Result 19: took 19 steps to catch monster!
Result 20: took 36 steps to catch monster!
Result 21: took 31 steps to catch monster!
Result 22: took 23 steps to catch monster!
Result 23: took 12 steps to catch monster!
Result 24: took 17 step

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

1. Chiến thuật: Thợ săn lưu trữ vị trí của quái vật vào bộ nhớ nội bộ (state). Nếu nó không biết quái vật ở đâu, nó sẽ gọi hành động "listen". Sau khi lấy được tọa độ, nó sẽ luôn tính toán khoảng cách Manhattan để đi thẳng đến mục tiêu. Khi đến nơi mà quái vật đã bỏ đi, nó tiếp tục "listen" để cập nhật tọa độ mới. Chiến thuật này hoạt động vô cùng hiệu quả vì nó loại bỏ triệt để các bước đi mù quáng vô ích, đảm bảo tác tử luôn rút ngắn khoảng cách với mục tiêu sau mỗi bước.

2. Teleportation:  không sử dụng teleport. Vì tính năng dịch chuyển đưa thợ săn đến một ô hoàn toàn ngẫu nhiên (np.random.randint). Trong bản đồ kích thước vừa phải như $30 \times 30$, việc tự bước đi đến một điểm xác định chắc chắn và nhanh hơn nhiều so với việc đánh cược vào xác suất ngẫu nhiên hạ cánh trúng gần quái vật. Teleport chỉ hữu dụng trong một lưới khổng lồ (ví dụ $1000 \times 1000$) khi khoảng cách đi bộ tốn nhiều bước hơn cả max_steps.

3. Kích thước bản đồ: Khi kích thước bản đồ tăng (từ $10 \times 10$ lên $30 \times 30$), số bước trung bình (Average steps) cũng sẽ tăng lên theo tỷ lệ thuận. Vì agent sử dụng khoảng cách Manhattan, số bước tốn thêm chính bằng khoảng cách địa lý sinh ra do map to ra. Tuy nhiên, nó vẫn hiếm khi fail, chứng tỏ thuật toán rất ổn định với mọi kích cỡ không gian.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

Thay đổi thuật toán cho Quái vật:
Trong code hiện tại, hàm monster_agent_function_simple nhận trực tiếp tọa độ của cả hunter_pos và monster_pos trong mọi lượt đi mà không cần phải tốn bước để "listen". Thế nhưng quái vật lại chỉ đi random và có xác suất $50\%$ đứng im.

Để quái vật sống sót lâu hơn, ta có thể cải thiện nó thành một Reflex Agent thông minh:

- Chạy trốn thay vì đứng im: Loại bỏ hoàn toàn hành động "stay" (xác suất $50\%$). Việc đứng im chỉ giúp thợ săn bắt kịp nhanh hơn.

- Đi theo hướng ngược lại: Quái vật nên tính toán sự chênh lệch tọa độ tương tự như thợ săn. Thay vì thu hẹp khoảng cách, nó phải chọn hành động sao cho khoảng cách trục x,y ngày càng lớn ra.

Né góc: Khi quái vật chạm biên của map (VD monster_pos[0] == 0 là vướng tường), nó cần biết đánh lái qua lại thay vì cố gắng đi về hướng tường và bị kẹt, tạo cơ hội cho thợ săn áp sát dồn vào chân tường.

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.